In [3]:
import pandas as pd
import numpy as np

np.random.seed(42)
N = 3000

nationalities = np.random.choice(
    ['UAE National', 'Expat - South Asia', 'Expat - Arab', 'Expat - Western', 'Expat - East Asia'],
    N, p=[0.15, 0.40, 0.20, 0.10, 0.15]
)
genders   = np.random.choice(['Male', 'Female'], N, p=[0.55, 0.45])
ages      = np.random.normal(47, 13, N).clip(18, 85).astype(int)
emirates  = np.random.choice(
    ['Abu Dhabi', 'Dubai', 'Sharjah', 'Ajman', 'Other'],
    N, p=[0.35, 0.38, 0.14, 0.07, 0.06]
)

# ── Payer names matching Projects 1 & 2 ──────────────────────
payers = np.random.choice(
    ['Daman (HAAD)', 'AXA Gulf', 'BUPA Global', 'MetLife', 'ADNIC', 'Cigna', 'Sukoon Insurance'],
    N, p=[0.25, 0.18, 0.16, 0.14, 0.12, 0.09, 0.06]
)

plan_tiers = np.random.choice(['Basic', 'Enhanced', 'Premium'], N, p=[0.45, 0.35, 0.20])
conditions = np.random.choice(
    ['Diabetes', 'Hypertension', 'Cardiac', 'Diabetes + Hypertension', 'Diabetes + Cardiac'],
    N, p=[0.30, 0.28, 0.15, 0.17, 0.10]
)

bmi           = np.random.normal(29.5, 5.5, N).clip(17, 52).round(1)
systolic_bp   = np.random.normal(138, 22, N).clip(90, 200).astype(int)
smoker        = np.random.choice([0, 1], N, p=[0.72, 0.28])
comorbidities = np.random.poisson(1.4, N).clip(0, 5)
hba1c         = np.where(
    np.isin(conditions, ['Diabetes', 'Diabetes + Hypertension', 'Diabetes + Cardiac']),
    np.random.normal(8.2, 1.8, N).clip(5.5, 14.0), np.nan
).round(1)

er_visits         = np.random.poisson(1.6, N).clip(0, 8)
inpatient_days    = np.random.poisson(2.1, N).clip(0, 30)
outpatient_visits = np.random.poisson(6.5, N).clip(0, 30)
specialist_visits = np.random.poisson(3.2, N).clip(0, 15)

base_cost = {
    'Diabetes': 18000, 'Hypertension': 12000, 'Cardiac': 32000,
    'Diabetes + Hypertension': 26000, 'Diabetes + Cardiac': 42000
}
base       = np.array([base_cost[c] for c in conditions], dtype=float)
age_mult   = 1 + (ages - 40) * 0.012
bmi_mult   = 1 + (bmi - 25) * 0.018
smoke_mult = np.where(smoker == 1, 1.22, 1.0)
er_mult    = 1 + er_visits * 0.08
ip_mult    = 1 + inpatient_days * 0.06
comor_mult = 1 + comorbidities * 0.09
noise      = np.random.lognormal(0, 0.18, N)

total_cost_aed    = (base * age_mult * bmi_mult * smoke_mult * er_mult * ip_mult * comor_mult * noise).round(0)
pharmacy_cost     = (total_cost_aed * np.random.uniform(0.22, 0.38, N)).round(0)
procedure_cost    = (total_cost_aed * np.random.uniform(0.18, 0.35, N)).round(0)
consultation_cost = total_cost_aed - pharmacy_cost - procedure_cost

p33            = np.percentile(total_cost_aed, 33)
p66            = np.percentile(total_cost_aed, 66)
risk_tier      = np.where(total_cost_aed <= p33, 'Low', np.where(total_cost_aed <= p66, 'Medium', 'High'))
high_cost_flag = (total_cost_aed >= np.percentile(total_cost_aed, 80)).astype(int)

medication_compliance = np.random.choice(['High', 'Medium', 'Low'], N, p=[0.38, 0.37, 0.25])
member_ids = [f'MBR-{str(i+1).zfill(5)}' for i in range(N)]
years      = np.random.choice([2023, 2024], N, p=[0.48, 0.52])

df = pd.DataFrame({
    'member_id': member_ids, 'year': years, 'emirate': emirates,
    'nationality': nationalities, 'gender': genders, 'age': ages,
    'plan_tier': plan_tiers, 'payer': payers, 'condition': conditions,
    'bmi': bmi, 'hba1c': hba1c, 'systolic_bp': systolic_bp,
    'smoker': smoker, 'comorbidities': comorbidities,
    'medication_compliance': medication_compliance,
    'er_visits': er_visits, 'inpatient_days': inpatient_days,
    'outpatient_visits': outpatient_visits, 'specialist_visits': specialist_visits,
    'total_cost_aed': total_cost_aed.astype(int),
    'pharmacy_cost_aed': pharmacy_cost.astype(int),
    'procedure_cost_aed': procedure_cost.astype(int),
    'consultation_cost_aed': consultation_cost.astype(int),
    'risk_tier': risk_tier, 'high_cost_flag': high_cost_flag
})

df.to_csv('../data/chronic_disease_cohort.csv', index=False)
print(f"Dataset saved — {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nPayers: {sorted(df['payer'].unique())}")
df.head()

Dataset saved — 3,000 rows, 25 columns

Payers: ['ADNIC', 'AXA Gulf', 'BUPA Global', 'Cigna', 'Daman (HAAD)', 'MetLife', 'Sukoon Insurance']


,member_id,year,emirate,nationality,gender,age,plan_tier,payer,condition,bmi,...,er_visits,inpatient_days,outpatient_visits,specialist_visits,total_cost_aed,pharmacy_cost_aed,procedure_cost_aed,consultation_cost_aed,risk_tier,high_cost_flag
0,MBR-00001,2024,Abu Dhabi,Expat - South Asia,Female,33,Basic,Daman (HAAD),Diabetes,35.0,...,2,5,7,5,22859,7714,4122,11023,Low,0
1,MBR-00002,2023,Abu Dhabi,Expat - East Asia,Female,53,Basic,Sukoon Insurance,Diabetes + Cardiac,42.4,...,1,4,2,7,112306,26248,28582,57476,High,1
2,MBR-00003,2023,Dubai,Expat - Arab,Male,50,Basic,BUPA Global,Diabetes + Cardiac,25.9,...,1,2,8,2,69723,19232,23682,26809,High,1
3,MBR-00004,2024,Abu Dhabi,Expat - Arab,Female,59,Premium,AXA Gulf,Hypertension,38.7,...,1,2,10,2,29768,6781,9216,13771,Medium,0
4,MBR-00005,2023,Abu Dhabi,Expat - South Asia,Female,36,Enhanced,AXA Gulf,Cardiac,25.2,...,3,1,15,7,53033,12944,11720,28369,High,0


In [2]:
# Cell 2 — Dataset summary
print("=" * 55)
print("COHORT OVERVIEW")
print("=" * 55)
print(f"Total members:       {len(df):,}")
print(f"Total cost (AED):    {df['total_cost_aed'].sum():,.0f}")
print(f"Avg cost per member: AED {df['total_cost_aed'].mean():,.0f}")
print(f"Median cost:         AED {df['total_cost_aed'].median():,.0f}")
print(f"High-cost members:   {df['high_cost_flag'].sum():,} ({df['high_cost_flag'].mean()*100:.0f}%)")

print("\nCondition breakdown:")
print(df['condition'].value_counts().to_string())

print("\nRisk tier breakdown:")
print(df['risk_tier'].value_counts().to_string())

print("\nEmirate breakdown:")
print(df['emirate'].value_counts().to_string())

print("\nPayer breakdown:")
print(df['payer'].value_counts().to_string())

COHORT OVERVIEW
Total members:       3,000
Total cost (AED):    120,797,549
Avg cost per member: AED 40,266
Median cost:         AED 34,901
High-cost members:   600 (20%)

Condition breakdown:
condition
Diabetes                   876
Hypertension               832
Diabetes + Hypertension    502
Cardiac                    456
Diabetes + Cardiac         334

Risk tier breakdown:
risk_tier
High      1020
Low        990
Medium     990

Emirate breakdown:
emirate
Dubai        1126
Abu Dhabi    1050
Sharjah       416
Ajman         230
Other         178

Payer breakdown:
payer
DAMAN             855
AXA Gulf          605
MetLife           492
Oman Insurance    404
ADNIC             368
Al Buhaira        276
